In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,r2_score
import xgboost as xgb
from xgboost import XGBRegressor
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv('../data/dataset.csv')


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   traffic_volume       48204 non-null  int64  
 1   holiday              61 non-null     str    
 2   temp                 48204 non-null  float64
 3   rain_1h              48204 non-null  float64
 4   snow_1h              48204 non-null  float64
 5   clouds_all           48204 non-null  int64  
 6   weather_main         48204 non-null  str    
 7   weather_description  48204 non-null  str    
 8   date_time            48204 non-null  str    
dtypes: float64(3), int64(2), str(4)
memory usage: 4.8 MB


In [ ]:
# Convert datetime
df['date_time'] = pd.to_datetime(df['date_time'], dayfirst=True)
# Time features
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek
# Cyclic encoding for hour
import numpy as np
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
# Peak hour feature
df['is_peak_hour'] = df['hour'].apply(
    lambda x: 1 if (7 <= x <= 10 or 16 <= x <= 19) else 0
)
# Holiday encoding
df["is_holiday"] = df["holiday"].apply(lambda x: 0 if x == "None" else 1)
# Sort chronologically
df = df.sort_values('date_time')
# Lag features
df['lag_1'] = df['traffic_volume'].shift(1)
df["lag_2"] = df["traffic_volume"].shift(2)
df["lag_24"] = df["traffic_volume"].shift(24)
df["lag_168"] = df["traffic_volume"].shift(168)
# Rolling features (shift first to prevent leakage)
df["rolling_3"] = df["traffic_volume"].shift(1).rolling(3).mean()
df["rolling_6"] = df["traffic_volume"].shift(1).rolling(6).mean()
print("Before selective drop:", df.shape)
# Drop ONLY rows where engineered features are NaN
df = df.dropna(subset=[
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_3",
    "rolling_6"
])
print("After selective drop:", df.shape)
df

Before selective drop: (48204, 21)
After selective drop: (48036, 21)


,traffic_volume,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,hour,...,hour_sin,hour_cos,is_peak_hour,is_holiday,lag_1,lag_2,lag_24,lag_168,rolling_3,rolling_6
168,3418,NaN,288.86,0.0,0.0,75,Clouds,broken clouds,2012-10-09 19:00:00,19,...,-0.965926,0.258819,1,1,4460.0,6127.0,4259.0,5545.0,5645.333333,5507.166667
169,2775,NaN,287.36,0.0,0.0,90,Clouds,overcast clouds,2012-10-09 20:00:00,20,...,-0.866025,0.500000,0,1,3418.0,4460.0,3069.0,4516.0,4668.333333,5238.500000
170,2306,NaN,285.11,0.0,0.0,90,Clouds,overcast clouds,2012-10-09 21:00:00,21,...,-0.707107,0.707107,0,1,2775.0,3418.0,2378.0,4767.0,3551.000000,4817.666667
171,1846,NaN,283.46,0.0,0.0,90,Clouds,overcast clouds,2012-10-09 22:00:00,22,...,-0.500000,0.866025,0,1,2306.0,2775.0,2030.0,5026.0,2833.000000,4239.166667
172,947,NaN,282.45,0.0,0.0,90,Clouds,overcast clouds,2012-10-09 23:00:00,23,...,-0.258819,0.965926,0,1,1846.0,2306.0,1400.0,4918.0,2309.000000,3488.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48199,3543,NaN,283.45,0.0,0.0,75,Clouds,broken clouds,2018-09-30 19:00:00,19,...,-0.965926,0.258819,1,1,3947.0,4132.0,2950.0,4351.0,4120.666667,4224.333333
48200,2781,NaN,282.76,0.0,0.0,90,Clouds,overcast clouds,2018-09-30 20:00:00,20,...,-0.866025,0.500000,0,1,3543.0,3947.0,2607.0,4468.0,3874.000000,4084.833333
48201,2159,NaN,282.73,0.0,0.0,90,Thunderstorm,proximity thunderstorm,2018-09-30 21:00:00,21,...,-0.707107,0.707107,0,1,2781.0,3543.0,3856.0,4531.0,3423.666667,3831.333333
48202,1450,NaN,282.09,0.0,0.0,90,Clouds,overcast clouds,2018-09-30 22:00:00,22,...,-0.500000,0.866025,0,1,2159.0,2781.0,1826.0,4433.0,2827.666667,3474.166667


In [ ]:
X = df[[
    'hour_sin',
    'hour_cos',
    'day_of_week',
    'is_peak_hour',
    'is_holiday',
    'temp',
    'rain_1h',
    'snow_1h',
    'clouds_all',
    'lag_1',
    'lag_2',
    'lag_24',
    'lag_168',
    'rolling_3',
    'rolling_6',
]]
y = df["traffic_volume"]

In [6]:
# model = LinearRegression()
# model.fit(X_train, y_train)
# predictions = model.predict(X_test)
# print("MAE:", mean_absolute_error(y_test, predictions))
# print("R2:", r2_score(y_test, predictions))
# df[["date_time", "traffic_volume", "lag_1"]].head(10)

In [7]:
print("Shape after feature engineering:", df.shape)

Shape after feature engineering: (48036, 21)


In [ ]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_preds = lr.predict(X_test)
rf = RandomForestRegressor(n_estimators=300, max_depth=20, random_state=42)
rf.fit(X_train, y_train)

xg = xg

rf_preds = rf.predict(X_test)
print("Linear Regression Results:")
print("MAE:", mean_absolute_error(y_test, lr_preds))
print("R2:", r2_score(y_test, lr_preds))



print("\nRandom Forest Results:")
print("MAE:", mean_absolute_error(y_test, rf_preds))
print("R2:", r2_score(y_test, rf_preds))

Linear Regression Results:
MAE: 392.39656267170614
R2: 0.9248340173924487

Random Forest Results:
MAE: 154.7136605261414
R2: 0.983587126752868


In [ ]:
# import os
# import joblib
# os.makedirs("../models", exist_ok=True)
# joblib.dump(rf, "../models/TrafficAi-Model.pkl")

# print("Model saved ")

Model saved successfully ✅
